# 03 — SQL & Python Integration

## Objective

This notebook integrates Python and MySQL to reproduce selected business metrics identified during the exploratory analysis.

The objective is not to repeat the complete EDA performed in Python, but to demonstrate how SQL can be used to query relational data and generate business-ready metrics.

The workflow includes:

- loading the cleaned datasets required for the analysis;
- connecting Python to a MySQL database;
- creating SQL tables from pandas DataFrames;
- validating the SQL database;
- reproducing selected business metrics using SQL;
- importing SQL results back into Python;
- validating SQL results against the previous Python analysis;

## 1. Setup

The libraries required to connect Python with MySQL and manage the project datasets are imported below.

The MySQL password is requested securely at runtime so that database credentials are not stored in the notebook or uploaded to GitHub.

In [1]:
# Import libraries

import pandas as pd
from pathlib import Path
import getpass

from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

## 2. Load cleaned datasets

The SQL workflow uses the cleaned datasets exported from Notebook 01 rather than the original raw files.

Only the datasets required to reproduce the selected business metrics are loaded.

In [2]:
customers = pd.read_csv("../data/processed/customers_clean.csv")
orders = pd.read_csv("../data/processed/orders_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
reviews = pd.read_csv("../data/processed/reviews_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
category_translation = pd.read_csv("../data/processed/category_translation_clean.csv") 

mql = pd.read_csv("../data/processed/mql_clean.csv")
closed_deals = pd.read_csv("../data/processed/closed_deals_clean.csv")

### 2.1 Validate loaded datasets

A basic validation is performed before uploading the DataFrames to MySQL.

In [3]:
# Check dataset shapes

datasets = {
    "customers": customers,"orders": orders, "order_items": order_items,  "reviews": reviews, "products": products, "category_translation": category_translation, "mql": mql, "closed_deals": closed_deals}

for name, df in datasets.items():
    print(name, df.shape)

customers (99441, 5)
orders (99441, 8)
order_items (112650, 7)
reviews (99224, 5)
products (32951, 9)
category_translation (71, 2)
mql (8000, 4)
closed_deals (842, 14)


### 2.2 Prepare date columns for SQL

CSV files do not preserve pandas datetime data types. The date columns required for the SQL analysis are therefore converted back to datetime before uploading the tables to MySQL.

In [4]:
# Convert order dates to datetime

order_date_columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]

for column in order_date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

In [5]:
# Check order date types

orders[order_date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 3. Python → MySQL

The cleaned pandas DataFrames are uploaded to a MySQL database using SQLAlchemy.

This creates a relational SQL environment that can be used to reproduce selected business metrics from the previous Python analysis.

### 3.1 Upload cleaned datasets to MySQL

In [6]:
# MySQL connection settings

username = "root"
password = getpass.getpass("MySQL password: ")
host = "localhost"
database = "olist_project"

encoded_password = quote_plus(password)

MySQL password:  ········


In [7]:
# Connect to MySQL server

server_engine = create_engine(f"mysql+pymysql://{username}:{encoded_password}@{host}")

# Create project database if it does not exist

with server_engine.connect() as connection:
    connection.execute(text("CREATE DATABASE IF NOT EXISTS olist_project"))

print("Database ready.")

Database ready.


In [8]:
# Create connection to the Olist project database

engine = create_engine(f"mysql+pymysql://{username}:{encoded_password}@{host}/{database}")

### 3.2 Test database connection

The connection is tested before uploading the project datasets.

In [9]:
# Test MySQL connection

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Connected to:", result.scalar())

Connected to: olist_project


### 3.3 Upload cleaned datasets to MySQL

Each cleaned DataFrame is stored as a SQL table.

Existing project tables with the same names are replaced so that the database always reflects the current cleaned datasets.

In [10]:
# Upload cleaned datasets to MySQL

for table_name, df in datasets.items():
    
    df.to_sql(name=table_name, con=engine, if_exists="replace", index=False, chunksize=5000)
    
    print(f"{table_name}: uploaded")

customers: uploaded
orders: uploaded
order_items: uploaded
reviews: uploaded
products: uploaded
category_translation: uploaded
mql: uploaded
closed_deals: uploaded


In [11]:
# Check tables created in MySQL

tables_sql = pd.read_sql(text("SHOW TABLES;"), engine)

tables_sql

,Tables_in_olist_project
0,category_translation
1,closed_deals
2,customers
3,mql
4,order_items
5,orders
6,products
7,reviews


In [12]:
# Compare Python and SQL row counts

row_validation = []

for table_name, df in datasets.items():
    
    sql_count = pd.read_sql(text(f"""SELECT COUNT(*) AS row_count FROM {table_name};"""), engine)["row_count"].iloc[0]
    
    row_validation.append({"table": table_name, "python_rows": len(df), "sql_rows": sql_count})

row_validation = pd.DataFrame(row_validation)

row_validation

,table,python_rows,sql_rows
0,customers,99441,99441
1,orders,99441,99441
2,order_items,112650,112650
3,reviews,99224,99224
4,products,32951,32951
5,category_translation,71,71
6,mql,8000,8000
7,closed_deals,842,842


In [13]:
# Check whether row counts match

row_validation["match"] = (row_validation["python_rows"]== row_validation["sql_rows"])

row_validation

,table,python_rows,sql_rows,match
0,customers,99441,99441,True
1,orders,99441,99441,True
2,order_items,112650,112650,True
3,reviews,99224,99224,True
4,products,32951,32951,True
5,category_translation,71,71,True
6,mql,8000,8000,True
7,closed_deals,842,842,True


## 4. SQL Business Analysis

Selected business metrics identified during the Python EDA are reproduced using SQL.

The objective is to demonstrate relational joins, aggregations, conditional logic and CTEs while validating the consistency of the previous analytical findings.

### 4.1 Seller commercial performance

Seller commercial performance is reproduced by connecting the Marketing Funnel with converted sellers and their marketplace transactions.

Seller-level GMV, number of orders and AOV are calculated first. These metrics are then aggregated by acquisition channel.

In [14]:
# Q1 | Seller commercial performance by acquisition channel

query_q1 = text("""
WITH seller_metrics AS (
    
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS number_of_orders,
        SUM(price) AS gmv,
        SUM(price) / COUNT(DISTINCT order_id) AS aov
        
    FROM order_items
    
    GROUP BY seller_id
),

seller_acquisition AS (
    
    SELECT
        cd.seller_id,
        m.origin
        
    FROM closed_deals AS cd
    
    INNER JOIN mql AS m
        ON cd.mql_id = m.mql_id
)

SELECT
    sa.origin AS acquisition_channel,
    COUNT(DISTINCT sa.seller_id) AS number_of_sellers,
    ROUND(AVG(sm.gmv), 2) AS average_seller_gmv,
    ROUND(AVG(sm.number_of_orders), 2) AS average_orders_per_seller,
    ROUND(AVG(sm.aov), 2) AS average_seller_aov

FROM seller_acquisition AS sa

INNER JOIN seller_metrics AS sm
    ON sa.seller_id = sm.seller_id

WHERE sa.origin NOT IN ('unknown', 'missing')
  AND sa.origin IS NOT NULL

GROUP BY sa.origin

ORDER BY average_seller_gmv DESC;
""")

In [15]:
acquisition_performance_sql = pd.read_sql(query_q1, engine)

acquisition_performance_sql

,acquisition_channel,number_of_sellers,average_seller_gmv,average_orders_per_seller,average_seller_aov
0,other,2,3444.32,44.50,125.44
1,referral,9,1987.46,7.67,251.15
2,organic_search,113,1832.07,10.69,201.20
3,paid_search,101,1537.40,12.35,189.59
4,email,6,1414.16,3.67,406.97
5,social,31,1402.52,12.65,151.65
6,direct_traffic,31,706.58,6.23,124.68
7,display,2,461.50,3.50,129.00


### 4.2 Regional delivery performance

Delivery performance is reproduced by connecting delivered orders with customer location.

Two operational metrics are calculated by customer state:

- average delivery time;
- late delivery rate.

This reproduces the regional operational analysis from Q2.

In [16]:
# Q2 | Regional delivery performance

query_q2 = text("""
SELECT
    c.customer_state,
    
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    
    ROUND(
        AVG(
            DATEDIFF(
                o.order_delivered_customer_date,
                o.order_purchase_timestamp
            )
        ),
        2
    ) AS average_delivery_days,
    
    ROUND(
        AVG(
            CASE
                WHEN o.order_delivered_customer_date
                     > o.order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) * 100,
        2
    ) AS late_delivery_rate

FROM orders AS o

INNER JOIN customers AS c
    ON o.customer_id = c.customer_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY c.customer_state

ORDER BY average_delivery_days DESC;
""")

In [17]:
delivery_by_state_sql = pd.read_sql(query_q2, engine)

delivery_by_state_sql

,customer_state,delivered_orders,average_delivery_days,late_delivery_rate
0,RR,41,29.34,12.20
1,AP,67,27.18,4.48
2,AM,145,26.36,4.14
3,AL,397,24.50,23.93
4,PA,946,23.73,12.37
5,MA,717,21.51,19.67
6,SE,335,21.46,15.22
7,CE,1279,21.20,15.32
8,AC,80,21.00,3.75
9,PB,517,20.39,11.03


#### 4.3.1 Delivery reliability and customer satisfaction

Customer reviews may contain multiple records for the same order.

A CTE is used to create one average satisfaction score per order before connecting reviews with delivery performance. This maintains the order as the unit of analysis and reproduces the methodology used in the Python EDA.

In [18]:
# Q3 | Customer satisfaction by delivery reliability

query_q3_delivery = text("""
WITH order_reviews AS (
    
    SELECT
        order_id,
        AVG(review_score) AS review_score
        
    FROM reviews
    
    GROUP BY order_id
)

SELECT
    
    CASE
        WHEN o.order_delivered_customer_date
             > o.order_estimated_delivery_date
        THEN 'Late'
        ELSE 'On time'
    END AS delivery_status,
    
    COUNT(DISTINCT o.order_id) AS number_of_orders,
    
    ROUND(
        AVG(r.review_score),
        2
    ) AS average_review_score

FROM orders AS o

INNER JOIN order_reviews AS r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL

GROUP BY delivery_status

ORDER BY average_review_score DESC;
""")

In [19]:
delivery_satisfaction_sql = pd.read_sql(query_q3_delivery, engine)

delivery_satisfaction_sql

,delivery_status,number_of_orders,average_review_score
0,On time,88163,4.29
1,Late,7661,2.57


#### 4.3.2 Product category and customer satisfaction

To avoid duplicating customer reviews across multiple products, the SQL analysis follows the same unit-of-analysis rule used in Python.

The query:

1. assigns an English product category;
2. identifies the number of unique categories in each order;
3. retains orders associated with exactly one category;
4. creates one satisfaction score per order;
5. calculates the negative review rate by category;
6. retains categories with at least 100 reviewed orders.

In [20]:
# Q3 | Negative review rate by product category

query_q3_category = text("""
WITH translated_items AS (

    SELECT DISTINCT
        oi.order_id,
        
        CASE
            WHEN p.product_category_name =
                 'portateis_cozinha_e_preparadores_de_alimentos'
            THEN 'small_kitchen_appliances'
            
            WHEN p.product_category_name = 'pc_gamer'
            THEN 'pc_gamer'
            
            WHEN p.product_category_name = 'unknown'
            THEN 'unknown'
            
            ELSE COALESCE(
                ct.product_category_name_english,
                p.product_category_name
            )
        END AS product_category

    FROM order_items AS oi

    INNER JOIN products AS p
        ON oi.product_id = p.product_id

    LEFT JOIN category_translation AS ct
        ON p.product_category_name =
           ct.product_category_name
),

single_category_orders AS (

    SELECT
        order_id
        
    FROM translated_items
    
    GROUP BY order_id
    
    HAVING COUNT(DISTINCT product_category) = 1
),

order_categories AS (

    SELECT DISTINCT
        ti.order_id,
        ti.product_category
        
    FROM translated_items AS ti
    
    INNER JOIN single_category_orders AS sc
        ON ti.order_id = sc.order_id
),

order_reviews AS (

    SELECT
        order_id,
        AVG(review_score) AS review_score
        
    FROM reviews
    
    GROUP BY order_id
)

SELECT
    oc.product_category,
    
    COUNT(*) AS reviewed_orders,
    
    ROUND(
        AVG(r.review_score),
        2
    ) AS average_review_score,
    
    ROUND(
        AVG(
            CASE
                WHEN r.review_score <= 2
                THEN 1
                ELSE 0
            END
        ) * 100,
        2
    ) AS negative_review_rate

FROM order_categories AS oc

INNER JOIN order_reviews AS r
    ON oc.order_id = r.order_id

GROUP BY oc.product_category

HAVING COUNT(*) >= 100

ORDER BY negative_review_rate DESC;
""")

In [21]:
category_satisfaction_sql = pd.read_sql(query_q3_category, engine)

category_satisfaction_sql.head()

,product_category,reviewed_orders,average_review_score,negative_review_rate
0,fashion_male_clothing,110,3.73,25.45
1,office_furniture,1248,3.63,22.28
2,audio,341,3.85,21.41
3,construction_tools_safety,159,3.87,19.50
4,fixed_telephony,212,3.89,18.87


In [22]:
# Business overview for Tableau

business_overview_query = """
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, '%%Y-%%m') AS month,
    COUNT(DISTINCT o.order_id) AS number_of_orders,
    COUNT(DISTINCT o.customer_id) AS number_of_customers,
    ROUND(SUM(oi.price), 2) AS gmv,
    ROUND(
        SUM(oi.price) / COUNT(DISTINCT o.order_id),
        2
    ) AS average_order_value
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY month
ORDER BY month;
"""

business_overview_sql = pd.read_sql(business_overview_query, engine)

business_overview_sql

,month,number_of_orders,number_of_customers,gmv,average_order_value
0,2016-09,1,1,134.97,134.97
1,2016-10,265,265,40325.11,152.17
2,2016-12,1,1,10.90,10.90
3,2017-01,750,750,111798.36,149.06
4,2017-02,1653,1653,234223.40,141.70
5,2017-03,2546,2546,359198.85,141.08
6,2017-04,2303,2303,340669.68,147.92
7,2017-05,3546,3546,489338.25,138.00
8,2017-06,3135,3135,421923.37,134.58
9,2017-07,3872,3872,481604.52,124.38


## 5. SQL → Python

The SQL query results are now available as pandas DataFrames.

This demonstrates the reverse direction of the integration: SQL performs the relational transformations and aggregations, while Python can consume the resulting business-ready datasets for validation, additional analysis or visualization.

In [23]:
# SQL results available in Python

sql_outputs = {"business_overview": business_overview_sql, "acquisition_performance": acquisition_performance_sql, "delivery_by_state": delivery_by_state_sql, "delivery_satisfaction": delivery_satisfaction_sql, "category_satisfaction": category_satisfaction_sql}

for name, df in sql_outputs.items():
    print(name, df.shape)

business_overview (23, 5)
acquisition_performance (8, 5)
delivery_by_state (27, 4)
delivery_satisfaction (2, 3)
category_satisfaction (52, 4)


### 5.1 Validate SQL results against Python findings

The SQL outputs are compared with the main findings identified during the Python EDA.

The objective is to verify that both workflows produce consistent business conclusions.

In [24]:
# Business overview SQL result

business_overview_sql

,month,number_of_orders,number_of_customers,gmv,average_order_value
0,2016-09,1,1,134.97,134.97
1,2016-10,265,265,40325.11,152.17
2,2016-12,1,1,10.90,10.90
3,2017-01,750,750,111798.36,149.06
4,2017-02,1653,1653,234223.40,141.70
5,2017-03,2546,2546,359198.85,141.08
6,2017-04,2303,2303,340669.68,147.92
7,2017-05,3546,3546,489338.25,138.00
8,2017-06,3135,3135,421923.37,134.58
9,2017-07,3872,3872,481604.52,124.38


In [25]:
# Q1 SQL result

acquisition_performance_sql

,acquisition_channel,number_of_sellers,average_seller_gmv,average_orders_per_seller,average_seller_aov
0,other,2,3444.32,44.50,125.44
1,referral,9,1987.46,7.67,251.15
2,organic_search,113,1832.07,10.69,201.20
3,paid_search,101,1537.40,12.35,189.59
4,email,6,1414.16,3.67,406.97
5,social,31,1402.52,12.65,151.65
6,direct_traffic,31,706.58,6.23,124.68
7,display,2,461.50,3.50,129.00


In [26]:
# Q2 states with the longest average delivery times

delivery_by_state_sql.head(10)

,customer_state,delivered_orders,average_delivery_days,late_delivery_rate
0,RR,41,29.34,12.20
1,AP,67,27.18,4.48
2,AM,145,26.36,4.14
3,AL,397,24.50,23.93
4,PA,946,23.73,12.37
5,MA,717,21.51,19.67
6,SE,335,21.46,15.22
7,CE,1279,21.20,15.32
8,AC,80,21.00,3.75
9,PB,517,20.39,11.03


In [27]:
# Q2 states with the highest late delivery rates

delivery_by_state_sql.sort_values("late_delivery_rate", ascending=False).head(10)

,customer_state,delivered_orders,average_delivery_days,late_delivery_rate
3,AL,397,24.50,23.93
5,MA,717,21.51,19.67
10,PI,476,19.40,15.97
7,CE,1279,21.20,15.32
6,SE,335,21.46,15.22
11,BA,3256,19.28,14.04
21,RJ,12350,15.24,13.47
16,TO,274,17.60,12.77
4,PA,946,23.73,12.37
17,ES,1995,15.72,12.23


In [28]:
# Q3 delivery reliability and satisfaction

delivery_satisfaction_sql

,delivery_status,number_of_orders,average_review_score
0,On time,88163,4.29
1,Late,7661,2.57


In [29]:
# Q3 categories with the highest negative review rates

category_satisfaction_sql.head(15)

,product_category,reviewed_orders,average_review_score,negative_review_rate
0,fashion_male_clothing,110,3.73,25.45
1,office_furniture,1248,3.63,22.28
2,audio,341,3.85,21.41
3,construction_tools_safety,159,3.87,19.50
4,fixed_telephony,212,3.89,18.87
5,unknown,1379,3.96,18.56
6,fashion_underwear_beach,120,3.93,18.33
7,home_confort,345,3.95,17.39
8,air_conditioning,247,4.04,16.19
9,bed_bath_table,9111,3.99,16.17


### Validation result

The SQL analysis reproduces the main patterns identified during the Python EDA and also provides an executive overview of marketplace performance.

The results confirm that:

- marketplace activity can be summarized through monthly GMV, orders, customers and average order value;
- acquisition channels show descriptive differences in seller commercial performance;
- delivery performance varies substantially across customer states;
- late deliveries are associated with considerably lower customer review scores;
- negative customer reviews are concentrated in specific product categories.

This consistency provides an additional validation of the business metrics used in the project and prepares the final outputs for visualization in Tableau.

In [30]:
# Create Tableau output folder

TABLEAU_PATH = Path("../tableau")

TABLEAU_PATH.mkdir(parents=True, exist_ok=True)

In [31]:
# Export SQL results for Tableau

business_overview_sql.to_csv(TABLEAU_PATH / "tableau_business_overview.csv", index=False)

acquisition_performance_sql.to_csv(TABLEAU_PATH / "tableau_acquisition_performance.csv", index=False)

delivery_by_state_sql.to_csv(TABLEAU_PATH / "tableau_delivery_by_state.csv", index=False)

delivery_satisfaction_sql.to_csv(TABLEAU_PATH / "tableau_delivery_satisfaction.csv", index=False)

category_satisfaction_sql.to_csv(TABLEAU_PATH / "tableau_category_satisfaction.csv", index=False)

print("Tableau datasets exported successfully.")

Tableau datasets exported successfully.


In [32]:
# Check Tableau output files

for file in TABLEAU_PATH.glob("*.csv"): print(file.name)

tableau_acquisition_performance.csv
tableau_business_overview.csv
tableau_category_satisfaction.csv
tableau_delivery_by_state.csv
tableau_delivery_satisfaction.csv


##  6. Conclusion

This notebook successfully integrated Python and MySQL to reproduce selected business metrics from the previous exploratory analysis.

The cleaned datasets were uploaded from pandas to MySQL, validated through row-count checks, and queried using SQL techniques such as joins, aggregations, conditional logic, date functions, `HAVING` clauses and CTEs.

The SQL results were then imported back into Python using `pd.read_sql()` and compared with the findings from the previous EDA.

The results were consistent across both workflows:

- acquisition channels show descriptive differences in seller commercial performance;
- delivery performance varies substantially across Brazilian states;
- late deliveries are associated with considerably lower customer satisfaction;
- negative reviews are concentrated in specific product categories.

This consistency provides an additional validation of the business metrics used in the project and demonstrates the integration of Python and SQL within an end-to-end analytics workflow. The validated SQL outputs were also exported as aggregated, business-ready datasets for the final Tableau visualization stage.